# 00c - LISFLOOD domain mask on the ERA5 0.25 deg grid

**Purpose**: build, once, the mask that is applied to every 0.25 deg output (GCM "ERA5-like" after the downscaling,
ERA5 for the validation, later the RCM), so that every dataset is compared on exactly the same pixels and LISFLOOD
gets all the pixels it needs.

**Input**: the GloFAS / LISFLOOD domain `template_Global_03min.nc` (0.05 deg, 1 = inside the domain, NaN = outside;
90N-60S, no Antarctica; built on the MERIT Hydro river network), downloaded from the JRC open-data FTP:
`CEMS-GLOFAS/LISFLOOD_static_and_parameter_maps_for_GloFAS/v2.1.1_OS-LISFLOOD-v5.x/Main/`, stored in `extra_files/`.

**Rule ("at least one")**: an ERA5 pixel is inside the domain if **at least one** of the 0.05 deg LISFLOOD pixels whose
centre falls inside it is inside the domain (about 5 x 5 LISFLOOD pixels per ERA5 pixel). Every LISFLOOD pixel of the
domain is therefore covered, plus a thin coastal margin.

**Output** (`extra_files/lisflood_domain_on_ERA5_025.nc`), on the ERA5 grid as read by the pipeline
(ascending latitude, longitude -180..179.75):
- `domain` (1 = inside, 0 = outside)
- `domain_frac`: share of the LISFLOOD pixels of each ERA5 pixel that are inside the domain (information only)

Use it with `ba_utils.load_domain_mask(cfg)` (ERA5 grid), or `ba_utils.load_domain_mask(cfg, like=da)` to transfer it
to another grid (e.g. native EC-Earth3, for the exploration) with the same rule.

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, warnings, datetime
from pathlib import Path
for _p in (Path.cwd(), Path.cwd().parent):   # ba_utils.py: same folder as the notebook, or the parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
cfg = bu.load_config(CONFIG_PATH)

EXTRA = bu.domain_mask_path(cfg).parent
LISFLOOD_FILE = EXTRA / "template_Global_03min.nc"
OUT_FILE = bu.domain_mask_path(cfg)
print("input :", LISFLOOD_FILE, "| exists:", LISFLOOD_FILE.exists())
print("output:", OUT_FILE)

## 1. Inputs: LISFLOOD domain (0.05 deg) and ERA5 grid (0.25 deg)

In [ ]:
lis = xr.open_dataset(LISFLOOD_FILE)
lis = bu.norm_coords(lis[list(lis.data_vars)[0]]).load()       # ascending lat, lon -180..180
fine = (lis == 1).values
print(f"LISFLOOD: {lis.sizes['lat']} x {lis.sizes['lon']}, lat {float(lis.lat.min()):.3f}..{float(lis.lat.max()):.3f}, "
      f"lon {float(lis.lon.min()):.3f}..{float(lis.lon.max()):.3f} | pixels in the domain: {fine.sum():,}")

# ERA5 grid: only the coordinates of ONE ERA5 file are needed (all years share the same grid),
# normalised as the pipeline reads it (same coordinates as every ERA5 / ERA5-like field)
ERA5_FILE = bu._pattern_files(cfg["datasets"]["era5"]["patterns"]["pr"])[0]
era5 = bu.norm_coords(xr.open_dataset(ERA5_FILE)["pr"].isel(time=0))
print("ERA5 grid taken from:", ERA5_FILE)
print(f"ERA5    : {era5.sizes['lat']} x {era5.sizes['lon']}, lat {float(era5.lat.min())}..{float(era5.lat.max())}, "
      f"lon {float(era5.lon.min())}..{float(era5.lon.max())}")

## 2. Transfer to the ERA5 grid (rule: at least one LISFLOOD pixel inside)

In [ ]:
mask, frac = bu.mask_any(fine, lis.lat.values, lis.lon.values, era5.lat.values, era5.lon.values)
domain = xr.DataArray(mask.astype("int8"), dims=("lat", "lon"), coords={"lat": era5.lat, "lon": era5.lon}, name="domain")
domain_frac = xr.DataArray(frac.astype("float32"), dims=("lat", "lon"), coords={"lat": era5.lat, "lon": era5.lon}, name="domain_frac")

# checks: every LISFLOOD pixel of the domain falls in an ERA5 pixel of the mask
def globe_share(m, lat, res):
    """Share of the area of the whole sphere covered by the True cells of a regular grid of step `res` deg."""
    return float((np.cos(np.deg2rad(lat))[:, None] * m).sum() * np.deg2rad(res) ** 2 / (4 * np.pi))

area_era5 = globe_share(mask, era5.lat.values, 0.25)
area_lis = globe_share(fine, lis.lat.values, 0.05)
iy = np.searchsorted(bu._edges(era5.lat.values), lis.lat.values) - 1
xe = bu._edges(era5.lon.values)
ix = np.clip(np.searchsorted(xe, ((lis.lon.values - xe[0]) % 360) + xe[0]) - 1, 0, era5.sizes["lon"] - 1)
r, c = np.nonzero(fine)
missed = int((~mask[iy[r], ix[c]]).sum())
print(f"ERA5 pixels in the domain: {mask.sum():,} ({100 * area_era5:.1f} % of the globe area)")
print(f"LISFLOOD domain           : {100 * area_lis:.1f} % of the globe area (the ERA5 mask is a bit larger: coastal margin)")
print(f"LISFLOOD pixels of the domain NOT covered by the ERA5 mask: {missed} (must be 0)")
assert missed == 0

## 3. Save

In [ ]:
out = xr.Dataset({"domain": domain, "domain_frac": domain_frac})
out["domain"].attrs = {"long_name": "LISFLOOD (GloFAS) domain on the ERA5 0.25 deg grid", "flag_values": "0 1",
                       "flag_meanings": "outside inside"}
out["domain_frac"].attrs = {"long_name": "share of the 0.05 deg LISFLOOD pixels of the ERA5 pixel that are in the domain",
                            "units": "1"}
out.attrs = {"source": str(LISFLOOD_FILE.name) + " (GloFAS LISFLOOD static maps v2.1.1_OS-LISFLOOD-v5.x, Main/)",
             "rule": "ERA5 pixel inside if at least one LISFLOOD pixel whose centre falls in it is inside",
             "grid": "ERA5 0.25 deg as read by ba_utils.read_raw (ascending lat, lon -180..179.75)",
             "created": datetime.date.today().isoformat(), "created_by": "notebook 00c_land_mask"}
tmp = OUT_FILE.with_suffix(".tmp.nc")
out.to_netcdf(tmp, encoding={"domain": {"zlib": True}, "domain_frac": {"zlib": True}})
tmp.replace(OUT_FILE)
print("written:", OUT_FILE)
m = bu.load_domain_mask(cfg)
print("reload check:", int(m.sum()), "pixels | same grid as ERA5:", bu.same_grid(m, era5))

## 4. Control maps

Grey = outside, colour = inside. Check in particular the large lakes (Caspian Sea, Great Lakes, Lake Victoria),
the coasts, small islands and Greenland.

In [ ]:
def show(ax, da, title, extent=None):
    d = da if extent is None else da.sel(lat=slice(extent[2], extent[3]), lon=slice(extent[0], extent[1]))
    ax.imshow(np.where(np.isfinite(d.values.astype(float)), d.values.astype(float), np.nan), origin="lower",
              extent=[float(d.lon.min()), float(d.lon.max()), float(d.lat.min()), float(d.lat.max())],
              cmap="viridis", vmin=0, vmax=1, interpolation="nearest", aspect="auto")
    ax.set_facecolor("0.85"); ax.set_title(title, fontsize=9)

lis_plot = lis.where(lis == 1)                                  # NaN outside
dom_plot = domain.where(domain == 1).astype(float)
fig, axes = plt.subplots(1, 2, figsize=(18, 4.5))
show(axes[0], lis_plot[::5, ::5], "LISFLOOD domain (0.05 deg, shown 1 pixel out of 5)")
show(axes[1], dom_plot, "LISFLOOD domain on the ERA5 0.25 deg grid")
plt.tight_layout(); plt.savefig(EXTRA / "lisflood_domain_on_ERA5_025_global.png", dpi=110); plt.show()

ZOOMS = {"Europe": (-25, 45, 30, 72), "Caspian Sea": (45, 56, 36, 48), "Great Lakes": (-93, -75, 40, 50),
         "Lake Victoria": (30, 36, -4, 2), "Greenland": (-75, -10, 58, 84)}
fig, axes = plt.subplots(2, len(ZOOMS), figsize=(4.2 * len(ZOOMS), 7))
for c, (lab, ext) in enumerate(ZOOMS.items()):
    show(axes[0, c], lis_plot, f"{lab}: LISFLOOD 0.05 deg", ext)
    show(axes[1, c], dom_plot, f"{lab}: on ERA5 0.25 deg", ext)
plt.tight_layout(); plt.savefig(EXTRA / "lisflood_domain_on_ERA5_025_zooms.png", dpi=110); plt.show()

## 5. How to use it

```python
m = bu.load_domain_mask(cfg)              # ERA5 0.25 deg grid (True = inside)
x_land = x.where(m)                       # x: any field on the ERA5 grid (ERA5, GCM ERA5-like 0.25 deg, ...)
m7 = bu.load_domain_mask(cfg, like=gcm)   # same mask transferred to the native EC-Earth3 grid ("at least one" rule)
```
The downscaling itself runs on the whole globe; the mask is applied to its outputs and to the validation.